# Phase 1: Offline Distillation Data Generation (API Version)
Notebook này sinh dữ liệu ma trận khoảng cách $D$ giữa các rollouts giải toán để phục vụ quá trình Distillation cho Student Model.
**Được tối ưu hóa:** Gọi thẳng tới OpenRouter API cho Llama-3.1-8B-Instruct. KHÔNG cần GPU, chạy trực tiếp trên CPU cá nhân hoặc Colab Basic.

In [1]:
import os
import sys
# Ép vLLM dùng bản V0 và tắt các thông báo gây lỗi
os.environ["VLLM_USE_V1"] = "0"
os.environ["VLLM_NO_USAGE_STATS"] = "1"
# Vá lỗi fileno cho Colab
if not hasattr(sys.stdout, 'fileno'):
    sys.stdout.fileno = lambda: 1
if not hasattr(sys.stderr, 'fileno'):
    sys.stderr.fileno = lambda: 2

In [2]:
# [QUAN TRỌNG] Kết nối với Google Drive để đảm bảo lưu dữ liệu an toàn
import os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/Data_Phase1.5', exist_ok=True)
    os.makedirs('/content/drive/MyDrive/Data_Phase1.5/rollouts_cache', exist_ok=True)
    print("Đã kết nối Google Drive thành công!")
except:
    print("Không chạy trên môi trường Colab.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Đã kết nối Google Drive thành công!


In [3]:
!pip install -q datasets jsonlines tqdm openai transformers accelerate vllm

In [4]:
import torch, sys
print(torch.__version__)
print(torch.version.cuda)
print(sys.version_info)

2.11.0+cu130
13.0
sys.version_info(major=3, minor=12, micro=13, releaselevel='final', serial=0)


In [5]:
# !pip install flash-attn --no-build-isolation -q \
#   --find-links https://github.com/Dao-AILab/flash-attention/releases/expanded_assets/v2.8.3

In [6]:
# !git pull

In [7]:
!git clone https://github.com/cudnah124/GRPO-via-Real-Time-Trajectory-Alignment.git

fatal: destination path 'GRPO-via-Real-Time-Trajectory-Alignment' already exists and is not an empty directory.


In [8]:
%cd /content/GRPO-via-Real-Time-Trajectory-Alignment

/content/GRPO-via-Real-Time-Trajectory-Alignment


## Cấu hình API Key
Điền API Key của OpenRouter (hoặc Hugging Face) vào đây trước khi chạy.

In [ ]:
import os
import phase1_distillation.config as config
import importlib

# NHẬP CÁC TOKEN CỦA BẠN VÀO ĐÂY (CÁCH NHAU BẰNG DẤU PHẨY)
tokens_string = ''

os.environ['HF_TOKENS'] = tokens_string
os.environ['HF_TOKEN'] = tokens_string.split(',')[0] # Set thêm token lẻ cho datasets library

# Force reload config để nhận list token mới
importlib.reload(config)

print(f"Số lượng Token sẵn sàng xoay vòng: {len(config.HF_TOKENS)}")
if len(config.HF_TOKENS) > 0:
    print(f"Token đầu tiên: {config.HF_TOKENS[0][:10]}...")

Số lượng Token sẵn sàng xoay vòng: 3
Token đầu tiên: hf_UWNpIGF...


## Chạy Pipeline (Sinh K=4 Rollouts và Ma trận $D$)

In [10]:
import jsonlines
from tqdm import tqdm
from phase1_distillation import MathDataset, MathRolloutGenerator, AlignmentJudge
from phase1_distillation.dataset import get_problem_id
import phase1_distillation.config as config

DRIVE_OUTPUT_FILE = "/content/drive/MyDrive/Data_Phase1.5/distillation_data.jsonl"
config.DRIVE_OUTPUT_FILE = DRIVE_OUTPUT_FILE
config.DRIVE_CACHE_DIR = "/content/drive/MyDrive/Data_Phase1.5/rollouts_cache"
config.DRIVE_PROCESSED_IDS = "/content/drive/MyDrive/Data_Phase1.5/processed_ids.txt"
os.makedirs(os.path.dirname(config.DRIVE_OUTPUT_FILE), exist_ok=True)

# 1. Khởi tạo Dataset
# dataset, processed_ids = MathDataset.load_with_resume(config.DRIVE_PROCESSED_IDS, sample_size=100)

# 2. Khởi tạo Generator & Judge
generator = MathRolloutGenerator(model_id=config.GENERATOR_MODEL_ID)
judge = AlignmentJudge(model_id=config.JUDGE_MODEL_ID)

# # 3. Vòng lặp chính
# with jsonlines.open(config.DRIVE_OUTPUT_FILE, mode='a') as writer, open(config.DRIVE_PROCESSED_IDS, "a") as track_file:
#     for item in tqdm(dataset, total=len(dataset), desc="Generating Distillation Data"):
#         problem_id = get_problem_id(item['problem'])

#         # Sinh rollouts (CÓ CACHE)
#         rollouts = generator.generate(item['problem'], problem_id, cache_dir=config.DRIVE_CACHE_DIR)

#         if len(rollouts) < config.K_ROLLOUTS:
#             continue

#         # Chấm điểm (CÓ CACHE TỪNG CẶP)
#         distance_matrices = judge.evaluate_pairs(item['problem'], problem_id, rollouts, cache_dir=config.DRIVE_CACHE_DIR)

#         if distance_matrices is None or len(distance_matrices) < 6:
#             continue

#         # Ghi kết quả
#         result = {
#             "question_id": problem_id,
#             "question": item['problem'],
#             "ground_truth": item['ground_truth_solution'],
#             "generated_rollouts": rollouts,
#             "distance_matrices": distance_matrices
#         }
#         writer.write(result)

#         # Đánh dấu đã xong và dọn dẹp cache (cả rollout và judge)
#         track_file.write(problem_id + "\n")
#         track_file.flush()

#         for prefix in ["", "judge_"]:
#             f_name = f"{prefix}{problem_id}.json" if prefix == "judge_" else f"{problem_id}.json"
#             cache_path = os.path.join(config.DRIVE_CACHE_DIR, f_name)
#             if os.path.exists(cache_path):
#                 os.remove(cache_path)

# Cấu hình kích thước Batch (Số lượng bài toán xử lý cùng lúc)
PROBLEM_BATCH_SIZE = 4
# Tải danh sách ID đã xong
dataset, processed_ids = MathDataset.load_with_resume(
    config.DRIVE_PROCESSED_IDS,
    cache_dir=config.DRIVE_CACHE_DIR,
    sample_size=100
)
generator = MathRolloutGenerator(model_id=config.GENERATOR_MODEL_ID)
# 3. Vòng lặp chính
for i in range(0, len(dataset), PROBLEM_BATCH_SIZE):
    batch_items = dataset[i : i + PROBLEM_BATCH_SIZE]
    problems = batch_items['problem']
    problem_ids = [get_problem_id(p) for p in problems]

    # vLLM sẽ xử lý song song toàn bộ batch này cực nhanh
    generator.generate_batch(
        problems,
        problem_ids,
        cache_dir=config.DRIVE_CACHE_DIR
    )

    print(f"[*] Finished Batch {i//PROBLEM_BATCH_SIZE + 1}")


[*] Initializing vLLM engine (V0) with: Qwen/Qwen2.5-Math-1.5B-Instruct...
INFO 05-15 16:17:34 [utils.py:240] non-default args: {'trust_remote_code': True, 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'enforce_eager': True, 'model': 'Qwen/Qwen2.5-Math-1.5B-Instruct'}
WARNING 05-15 16:17:34 [envs.py:1866] Unknown vLLM environment variable detected: VLLM_USE_V1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


INFO 05-15 16:17:35 [model.py:568] Resolved architecture: Qwen2ForCausalLM
WARNING 05-15 16:17:35 [model.py:1982] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 05-15 16:17:35 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-15 16:17:35 [model.py:1697] Using max model len 4096
INFO 05-15 16:17:35 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-15 16:17:35 [vllm.py:886] Asynchronous scheduling is enabled.
WARNING 05-15 16:17:35 [vllm.py:942] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 05-15 16:17:35 [vllm.py:960] Inductor compilation was disabled by user settings, optimizations settings that are only active during inductor compilation will be ignored.
INFO 05-15 16:17:35 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPrior

(EngineCore pid=26971) Process EngineCore:
(EngineCore pid=26971) Traceback (most recent call last):
(EngineCore pid=26971)   File "/usr/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=26971)     self.run()
(EngineCore pid=26971)   File "/usr/lib/python3.12/multiprocessing/process.py", line 108, in run
(EngineCore pid=26971)     self._target(*self._args, **self._kwargs)
(EngineCore pid=26971)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1144, in run_engine_core
(EngineCore pid=26971)     raise e
(EngineCore pid=26971)   File "/usr/local/lib/python3.12/dist-packages/vllm/v1/engine/core.py", line 1114, in run_engine_core
(EngineCore pid=26971)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=26971)                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(EngineCore pid=26971)   File "/usr/local/lib/python3.12/dist-packages/vllm/tracing/otel.py", line 178, in sync_wr

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {}